# 00 — Colab Setup & Full-Pipeline Run

Run this notebook in [Google Colab](https://colab.research.google.com) (Runtime → Run all, or Cell → Run all)
to reproduce the **entire UK wheat forecasting pipeline** from scratch:

| Step | Stage | Notebook |
|------|-------|----------|
| 01 | Data Acquisition | `notebooks/01_Data_Acquisition.ipynb` |
| 02 | Modelling Table | `notebooks/02_Modelling_Table.ipynb` |
| 03 | EDA | `notebooks/03_EDA.ipynb` |
| 04 | Feature Engineering | `notebooks/04_Feature_Engineering.ipynb` |
| 05 | Model (CV, DM tests, PIs, oracle, verify) | `notebooks/05_Model.ipynb` |

Everything the pipeline needs — frozen raw data, canonical modelling table, and
expected thesis outputs — is committed to this repository, so the run works
**fully offline** once cloned (no downloads from Met Office required).


## 1. Clone the repository

Clones this repo into `/content/uk_wheat_pipeline` and moves into it. All
subsequent cells run with that directory as the working directory.


In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/Dickuta/msc-uk-wheat-forecast.git"
PROJECT_DIR = "/content/uk_wheat_pipeline"

os.makedirs("/content", exist_ok=True)
if not os.path.isdir(PROJECT_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, PROJECT_DIR],
        check=True,
    )
    print("Cloned repository to", PROJECT_DIR)
else:
    # The repo already exists on this VM (e.g. from an earlier run in the same
    # session). Force-pull the latest commit so requirements.txt and the data
    # bundle are never stale.
    subprocess.run(["git", "-C", PROJECT_DIR, "pull", "--ff-only"], check=True)
    print("Repository already present at", PROJECT_DIR, "- pulled latest.")

os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

## 2. Install dependencies

Installs the pinned versions from `requirements.txt` (numpy, pandas, scipy,
scikit-learn, statsmodels, matplotlib, seaborn, requests, prophet, xgboost).


In [ ]:
import os
import subprocess
import sys

# Colab pre-loads numpy into the kernel at startup (transitive dep of
# matplotlib). If we install a different numpy mid-session, the running kernel
# keeps the OLD numpy in sys.modules, so a later first-import of e.g.
# numpy._core.strings fails with "cannot import name '_center' from
# 'numpy._core.umath'". The only reliable fix is to restart the runtime once
# after the install so the new numpy is loaded cleanly.
#
# The /tmp flag persists across the restart, so "Run all" does not loop: on
# the second pass this cell detects the flag and just continues.
RESTART_FLAG = "/tmp/numpy_installed"

if not os.path.exists(RESTART_FLAG):
    print("Installing project dependencies from requirements.txt ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True,
    )

    # Colab pre-installs numpy; force a clean reinstall of the pinned build so
    # the .py files and compiled .so always match.
    print("Force-reinstalling pinned numpy to ensure a consistent build ...")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--force-reinstall",
            "--no-cache-dir",
            "numpy==2.3.5",
        ],
        check=True,
    )

    open(RESTART_FLAG, "w").close()
    print("Dependencies installed. Restarting the runtime to load the new numpy ...")
    os._exit(00)
else:
    print("Dependencies already installed in a clean runtime - continuing.")

In [ ]:
# numpy >= 2.4 removed `_blas_supports_fpe` from the compiled
# `_multiarray_umath`, but numpy.testing (pulled in via scipy/pandas
# on the src._bootstrap import path) still references it on import.
# requirements.txt pins numpy 2.3.5 (the last version that ships the
# attribute), so this is only a belt-and-braces guard. If a newer numpy
# somehow lands, inject a no-op stub matching the old behaviour
# (BLAS_SUPPORTS_FPE = False).
import numpy._core._multiarray_umath as _mu
if not hasattr(_mu, "_blas_supports_fpe"):
    _mu._blas_supports_fpe = lambda _: False
    print("Applied numpy _blas_supports_fpe compatibility shim.")
else:
    print("numpy _blas_supports_fpe present - no shim needed.")


## 3. Verify the data bundle

The raw weather files, the modelling table and the expected thesis outputs are
committed to the repo. Stage 01 detects the frozen raw files and skips the
network. This cell just confirms the data tree is intact.


In [ ]:
import os

from pathlib import Path

checks = {
    "data/raw/met_office_Tmean_UK.txt": "raw temperature source",
    "data/raw/met_office_Rainfall_UK.txt": "raw rainfall source",
    "data/raw/manifest.csv": "provenance manifest",
    "data/processed/uk_wheat_modelling_table_1980_2024.csv": "modelling table",
    "data/expected/model_comparison_results_corrected.csv": "expected results",
}
for path, label in checks.items():
    ok = os.path.isfile(path)
    print(f"{'OK  ' if ok else 'MISS'} {label:36s} {path}")
    if not ok:
        raise FileNotFoundError(f"Expected committed data file is missing: {path}")

raw_size = sum(p.stat().st_size for p in Path("data/raw").glob("*"))
print(f"\nCommitted data/raw bundle: {raw_size / 1024:.1f} KiB")

## 5. (Optional) Download the executed notebooks

Zips the executed notebooks and any generated outputs, then triggers a browser
download in Colab. Runs only inside Colab; it is skipped if executed locally.


# 01 · Data Acquisition — acquire the raw data

**Goal.** Download the raw inputs from their **public data sources** into
`data/raw/`, record a provenance manifest, and aggregate the weather series
to the four phenological windows used in the study. No data is hard-coded in
this notebook — everything is pulled from disk by the downstream stages.

| Stage | Reads from | Writes to |
|---|---|---|
| 01 Data Acquisition | public data sources (internet) | `data/raw/` + manifest |
| 02 EDA | `data/processed/uk_wheat_modelling_table_1980_2024.csv` | figures |
| 03 Modelling Table | `data/raw/` + canonical building blocks | modelling table |
| 04 Feature Engineering | modelling table | model-ready inputs |
| 05 Model | modelling table | result CSVs + figures |

## Data sources used in this study

| Variable | Source | Licence |
|---|---|---|
| Weather (temperature, rainfall) | Met Office UK Climate Series — areal values from HadUK-Grid 1 km | Open Government Licence |
| Wheat yield | DEFRA / USDA FAS national series (canonical file, see `data_sources.md`) | Open Government Licence |
| Policy dummies (CAP reform, Ukraine) | Constructed from documented policy events | — |

The Met Office series are **area-weighted UK means** derived from the HadUK-Grid
1 km dataset. Two files are downloaded:

* `Tmean` — monthly, seasonal and annual **mean air temperature** for the UK (degC)
* `Rainfall` — monthly, seasonal and annual **total rainfall** for the UK (mm)

**Temporal alignment.** For harvest year `Y`, weather is aligned as:
autumn = Oct–Nov of `Y−1`, winter = Dec(`Y−1`)–Feb(`Y`), spring = Mar–May(`Y`),
grain fill = Jun–Aug(`Y`). This guarantees **no forward-looking information**
leaks into a training window.

## 1.1 Setup

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import hashlib
import logging
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests

from src._bootstrap import init_script

display = init_script(width=140, max_columns=40)

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s"
)
log = logging.getLogger(__name__)
import config

## 1.2 Download the Met Office monthly series

Each series is saved verbatim as downloaded (`.txt`), then parsed into a tidy
long-format table (`year`, `month`, value) and also stored as CSV. A SHA-256
checksum of the raw file is recorded so the provenance is auditable.

Once a raw file exists on disk it is **frozen**: subsequent runs reuse it
without hitting the network, so a dead or reformatted source URL can never
destroy reproducibility after the first successful acquisition.

In [2]:
def download_met_series(var_name, info):
    """Download one Met Office series, save the raw text, and parse to long format.

    Returns a dict with parsed DataFrame, file paths and a sha256 checksum.
    """
    url = info["url"]
    raw_path = config.RAW_DIR / info["raw_file"]
    if raw_path.exists():
        payload = raw_path.read_bytes()
        log.info(
            "Using frozen raw copy %s (sha256=%s...) - skipping download of %s",
            raw_path.name,
            hashlib.sha256(payload).hexdigest()[:12],
            url,
        )
    else:
        log.info("Downloading %s from %s", var_name, url)
        try:
            resp = requests.get(url, timeout=60)
            resp.raise_for_status()
        except requests.RequestException as exc:
            raise RuntimeError(
                f"Failed to download {var_name} from {url}: {exc}. "
                f"Place a frozen copy at {raw_path} to bypass the network."
            ) from exc
        payload = resp.content
        raw_path.write_bytes(payload)

    sha = hashlib.sha256(payload).hexdigest()
    text = payload.decode("utf-8")
    lines = text.strip().split("\n")
    columns = [c.strip().lower() for c in lines[5].strip().split()]
    month_cols = [
        "jan",
        "feb",
        "mar",
        "apr",
        "may",
        "jun",
        "jul",
        "aug",
        "sep",
        "oct",
        "nov",
        "dec",
    ]
    month_map = {m: i + 1 for i, m in enumerate(month_cols)}

    records = []
    for line in lines[6:]:
        if not line.strip():
            continue
        parts = line.strip().split()
        if len(parts) < 13:
            continue
        try:
            year = int(parts[0])
        except ValueError:
            continue
        for col_name, month_num in month_map.items():
            col_idx = columns.index(col_name)
            if col_idx >= len(parts):
                continue
            raw = parts[col_idx].strip()
            if raw == "---" or raw == "":
                continue
            try:
                val = float(raw)
            except ValueError:
                continue
            records.append({"year": year, "month": month_num, var_name: val})

    df = pd.DataFrame(records)
    csv_path = config.RAW_DIR / f"met_office_{var_name}_monthly.csv"
    df.to_csv(csv_path, index=False)
    log.info(
        "Parsed %d monthly records -> %s (sha256=%s...)",
        len(df),
        csv_path.name,
        sha[:12],
    )
    return {
        "variable": var_name,
        "df": df,
        "raw_path": raw_path,
        "csv_path": csv_path,
        "sha256": sha,
    }

## 1.3 Provenance manifest

A single `manifest.csv` records, for every raw file, the source URL, the
download timestamp and the SHA-256 checksum. This is the answer to the
question *"where did this number come from?"*.

In [3]:
def build_manifest(downloads):
    """Write the provenance manifest CSV and display it."""
    manifest_rows = []
    for var_name, dl in downloads.items():
        manifest_rows.append(
            {
                "variable": dl["variable"],
                "source": config.MET_OFFICE_SOURCES[var_name]["description"],
                "url": config.MET_OFFICE_SOURCES[var_name]["url"],
                "raw_file": dl["raw_path"].name,
                "parsed_file": dl["csv_path"].name,
                "downloaded_at_utc": datetime.fromtimestamp(
                    dl["raw_path"].stat().st_mtime, timezone.utc
                ).strftime("%Y-%m-%d %H:%M:%S"),
                "sha256": dl["sha256"],
                "n_monthly_rows": len(dl["df"]),
                "status": "OK",
            }
        )
    manifest = pd.DataFrame(manifest_rows)
    manifest_path = config.RAW_DIR / "manifest.csv"
    manifest.to_csv(manifest_path, index=False)
    display(manifest)
    return manifest

## 1.4 Aggregate monthly weather to phenological windows

Temperature is averaged within each window; rainfall is summed. The result is
the **UK-mean** seasonal weather table that 03 uses to reproduce the weather
building block from first principles.

In [4]:
# The seasonal alignment (autumn = Oct-Nov of Y-1, winter = Dec(Y-1)-Feb(Y),
# spring = Mar-May(Y), grain fill = Jun-Aug(Y)) lives in `src/weather.py` and
# is shared with stage 03, so the two can never drift apart.
from src.weather import aggregate_seasonal
from src.guards import assert_alignment, assert_alignment_spanning_year


def aggregate_seasonal_weather(downloads):
    """Aggregate monthly series to seasonal windows, spot-check, and save."""
    seasonal_dfs = []
    for var_name, dl in downloads.items():
        seasonal_dfs.append(
            aggregate_seasonal(
                dl["df"],
                var_name,
                config.SEASON_WINDOWS,
                config.MET_OFFICE_SOURCES[var_name]["agg"],
            )
        )
    seasonal_uk_mean = seasonal_dfs[0]
    for df in seasonal_dfs[1:]:
        seasonal_uk_mean = seasonal_uk_mean.merge(df, on="year", how="outer")
    seasonal_uk_mean = (
        seasonal_uk_mean[seasonal_uk_mean["year"].between(*config.MODEL_TABLE_YEARS)]
        .sort_values("year")
        .reset_index(drop=True)
    )

    # Spot-check seasonal alignment with the guards (FR-3 / F-1)
    ref_year = int(seasonal_uk_mean["year"].iloc[len(seasonal_uk_mean) // 2])

    # autumn = mean(Oct, Nov of Y-1) -> year_offset=-1, months=[10, 11]
    assert_alignment(
        downloads["tas"]["df"],
        seasonal_uk_mean.loc[seasonal_uk_mean["year"] == ref_year, "autumn_tas"].iloc[
            0
        ],
        ref_year,
        months=[10, 11],
        year_offset=-1,
        value_col="tas",
        agg="mean",
    )
    # spring = mean(Mar, Apr, May of Y) -> year_offset=0, months=[3, 4, 5]
    assert_alignment(
        downloads["tas"]["df"],
        seasonal_uk_mean.loc[seasonal_uk_mean["year"] == ref_year, "spring_tas"].iloc[
            0
        ],
        ref_year,
        months=[3, 4, 5],
        year_offset=0,
        value_col="tas",
        agg="mean",
    )
    # grainfill = mean(Jun, Jul, Aug of Y) -> year_offset=0, months=[6, 7, 8]
    assert_alignment(
        downloads["tas"]["df"],
        seasonal_uk_mean.loc[
            seasonal_uk_mean["year"] == ref_year, "grainfill_tas"
        ].iloc[0],
        ref_year,
        months=[6, 7, 8],
        year_offset=0,
        value_col="tas",
        agg="mean",
    )
    # winter = Dec(Y-1) + Jan-Feb(Y) -> boundary-spanning, use the spanning guard
    assert_alignment_spanning_year(
        downloads["tas"]["df"],
        seasonal_uk_mean.loc[seasonal_uk_mean["year"] == ref_year, "winter_tas"].iloc[
            0
        ],
        ref_year,
        first_month=12,
        second_year_months=[1, 2],
        value_col="tas",
        agg="mean",
    )
    print("Seasonal alignment spot-check: OK")

    seasonal_uk_mean.to_csv(config.WEATHER_SEASONAL_UK_MEAN_FILE, index=False)
    print(f"UK-mean seasonal weather: {len(seasonal_uk_mean)} rows")
    display(seasonal_uk_mean.head())
    return seasonal_uk_mean

## 1.5 What the raw download provides vs. what the thesis numbers need

Two weather building blocks exist for this study:

1. **UK-mean weather** — reproducible from the Met Office series downloaded
   above (this notebook).
2. **Canonical weather** — the values actually used in the thesis modelling
   table, which come from a **warmer regional extraction** (England-focused)
   made before this pipeline was finalised. They cannot be reconstructed from
   the public UK-mean series alone.

The pipeline therefore treats the **canonical modelling table** as ground
truth for the Results chapter (so every thesis number reproduces exactly) and
uses the downloaded UK-mean series to *demonstrate* the end-to-end
reproducibility path. Stage 03 quantifies the difference between the two.

## 1.6 Summary

In [5]:
def main():
    """Run the full stage 01: download, manifest, aggregate, verify, summarise."""
    config.RAW_DIR.mkdir(parents=True, exist_ok=True)

    downloads = {}
    for var_name, info in config.MET_OFFICE_SOURCES.items():
        downloads[var_name] = download_met_series(var_name, info)

    build_manifest(downloads)
    aggregate_seasonal_weather(downloads)

    print("\nRaw data now on disk:")
    for path in sorted(config.RAW_DIR.iterdir()):
        print(f"  {path.name:45s} {path.stat().st_size:>10,} bytes")

In [ ]:
if __name__ == "__main__":
    main()

# 02 · Modelling Table — assemble the modelling table

**Goal.** Combine the three building blocks — yield, weather and policy
dummies — into the single **modelling table** used by every downstream stage
(EDA, feature engineering, modelling). The table is written to
`data/processed/uk_wheat_modelling_table_1980_2024.csv` and is the **only**
data source stages 03, 04 and 05 read.

| Building block | File | Provenance |
|---|---|---|
| Yield (t/ha) | `data/processed/uk_wheat_yield_1980_2024.csv` | DEFRA / USDA national series (canonical) |
| Canonical weather | `data/processed/uk_wheat_weather_seasonal_canonical.csv` | regional (England-focused) extraction used for the thesis results |
| UK-mean weather | `data/processed/uk_wheat_weather_seasonal_uk_mean.csv` | rebuilt from the Met Office series in stage 01 |
| Policy dummies | `data/processed/uk_wheat_policy_dummies_1980_2024.csv` | constructed from documented policy events |

**Why two weather tables?** The thesis Results chapter reproduces exactly
only when the *canonical* weather is used (it was extracted from a warmer
regional dataset before this pipeline was finalised). To keep the pipeline
fully auditable we keep both: the canonical table is the ground truth, and
the UK-mean table demonstrates what is reproducible from the public sources
alone. This notebook builds the final table from the canonical block and
then quantifies how far the UK-mean reconstruction would diverge.

## 2.1 Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(
    0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
)

from src._bootstrap import init_script, common_imports

c = common_imports()
display = init_script(float_format=lambda v: f"{v:,.3f}")

## 2.2 Load the building blocks

In [ ]:
def load_inputs():
    """Load yield, canonical weather, and policy dummies; print summaries."""
    yield_df = c.pd.read_csv(c.config.YIELD_FILE)
    yield_df["year"] = yield_df["year"].astype(int)

    policy_df = c.pd.read_csv(c.config.POLICY_DUMMIES_FILE)
    policy_df["year"] = policy_df["year"].astype(int)

    canonical_weather = c.pd.read_csv(c.config.WEATHER_SEASONAL_CANONICAL_FILE)
    canonical_weather["year"] = canonical_weather["year"].astype(int)

    print(
        f"Yield: {len(yield_df)} rows ({yield_df['year'].min()}-{yield_df['year'].max()})"
    )
    print(f"Canonical weather: {len(canonical_weather)} rows")
    print(f"Policy dummies: {len(policy_df)} rows")
    display(yield_df.head())

    return (
        yield_df,
        canonical_weather.rename(columns=c.config.WEATHER_RENAME),
        policy_df,
    )

## 2.3 Merge into the modelling table

All three inputs are keyed on `year` and joined with an inner merge. The
canonical weather columns are renamed to the short names used across the
pipeline (`autumn_temp`, `autumn_rain`, …).

In [ ]:
def merge_table(yield_df, weather_df, policy_df):
    """Inner-join the three building blocks on year, sorted chronologically."""
    return (
        yield_df.merge(weather_df, on="year", how="inner")
        .merge(policy_df, on="year", how="inner")
        .sort_values("year")
        .reset_index(drop=True)
    )

## 2.4 Validation

A good modelling table must be complete, correctly typed and temporally
contiguous. The checks below fail loudly if anything is off — and a failing
table is never written to disk.

In [ ]:
def validate(modelling):
    """Assert the modelling table passes all structural checks."""
    checks = {
        "year range": (
            modelling["year"].min() == c.config.MODEL_TABLE_YEARS[0]
            and modelling["year"].max() == c.config.MODEL_TABLE_YEARS[1]
        ),
        "no missing values": int(modelling.isna().sum().sum()) == 0,
        "no duplicate years": int(modelling["year"].duplicated().sum()) == 0,
        "contiguous years": (modelling["year"].diff().dropna() == 1).all(),
        "correct column order": list(modelling[c.config.COVARIATE_COLS].columns)
        == c.config.COVARIATE_COLS,
    }
    for name, ok in checks.items():
        print(f"  [{'OK' if ok else 'FAIL'}] {name}")

    # Fail early: a bad table must never reach downstream stages
    assert all(checks.values()), "modelling table failed validation — see checks above"
    return checks

## 2.5 Reproducibility check — canonical vs UK-mean weather

To make the provenance gap explicit, we rebuild the UK-mean seasonal weather
(from stage 01) and compare it, season by season, with the canonical values.
Correlation with yield is shown for both, so it is easy to see that the two
views of "the same weather year" carry different signal strengths.

In [ ]:
def load_uk_mean_weather():
    """Prefer the fresh rebuild from stage 01, else the copied fallback."""
    from src.weather import aggregate_seasonal

    raw_monthly = sorted(c.config.RAW_DIR.glob("met_office_*_monthly.csv"))
    if raw_monthly:
        dfs = []
        for var in ["tas", "rainfall"]:
            df = c.pd.read_csv(c.config.RAW_DIR / f"met_office_{var}_monthly.csv")
            dfs.append(
                aggregate_seasonal(
                    df,
                    var,
                    c.config.SEASON_WINDOWS,
                    c.config.MET_OFFICE_SOURCES[var]["agg"],
                )
            )
        merged = dfs[0].merge(dfs[1], on="year", how="outer")
        return (
            merged[merged["year"].between(*c.config.MODEL_TABLE_YEARS)]
            .sort_values("year")
            .reset_index(drop=True)
        )
    return c.pd.read_csv(c.config.WEATHER_SEASONAL_UK_MEAN_FILE)


def compare_weather(modelling):
    """Quantify canonical-vs-UK-mean weather divergence."""
    uk_mean = load_uk_mean_weather().rename(columns=c.config.WEATHER_RENAME)
    weather_cols = [
        c
        for c in c.config.COVARIATE_COLS
        if c not in ("cap_1992", "cap_2005", "ukraine_2022")
    ]
    cmp_df = c.pd.DataFrame(
        {
            "variable": weather_cols,
            "canonical_mean": modelling[weather_cols].mean().values,
            "uk_mean_mean": uk_mean[weather_cols].mean().values,
            "canonical_minus_uk": (
                modelling[weather_cols].mean().values
                - uk_mean[weather_cols].mean().values
            ),
            "corr_with_yield_canonical": [
                modelling["yield_t_ha"].corr(modelling[c]) for c in weather_cols
            ],
            "corr_with_yield_uk_mean": [
                modelling["yield_t_ha"].corr(uk_mean[c]) for c in weather_cols
            ],
        }
    )
    display(cmp_df.round(3))

### Interpretation

The canonical (regional) weather is consistently **warmer** than the UK-mean
series (positive `canonical_minus_uk` for every temperature window) and
**drier** in every rainfall window — exactly what one expects from an
England-focused extraction next to a whole-UK average (England is warmer and
drier than the UK as a whole). The correlation with yield also differs between
the two. This is why the **canonical table is the source of record** for the
thesis: the exact Results chapter numbers can only be reproduced from it. The
UK-mean path remains fully scriptable above, so the whole chain *from raw
download to modelling table* is transparent and repeatable.

## 2.6 Summary

In [ ]:
def main():
    """Run the full stage 02: load, merge, validate, persist, compare."""
    yield_df, weather_df, policy_df = load_inputs()

    modelling = merge_table(yield_df, weather_df, policy_df)

    validate(modelling)

    # Only persist after validation passes
    modelling.to_csv(c.config.MODEL_TABLE_FILE, index=False)
    print(
        f"Modelling table written: {modelling.shape[0]} rows x {modelling.shape[1]} cols -> {c.config.MODEL_TABLE_FILE.name}"
    )

    print("\nSummary statistics of the modelling table:")
    display(modelling.describe().T)

    print("Missing values by column:")
    display(modelling.isna().sum())

    compare_weather(modelling)

    print("\nFinal modelling table written to:", c.config.MODEL_TABLE_FILE)
    print(
        f"  {modelling.shape[0]} rows x {modelling.shape[1]} cols, "
        f"years {modelling['year'].min()}-{modelling['year'].max()}"
    )
    print("Columns:", list(modelling.columns))

In [ ]:
if __name__ == "__main__":
    main()

# 03 · Exploratory Data Analysis (EDA)

**Goal.** Understand the modelling table before any forecasting model is
built: distributions, trends, missingness, correlations and the policy
events that shaped UK wheat yields.

**Input.** `data/processed/uk_wheat_modelling_table_1980_2024.csv`
(produced by stage 02).

**Output.** Charts shown inline below (no image files are saved).

Every plot below is generated by the code cell that precedes it — nothing is
pre-rendered, so rerunning this notebook regenerates everything.

## 3.1 Setup & data load

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import matplotlib.pyplot as plt
import seaborn as sns

from src._bootstrap import init_script, common_imports

c = common_imports()
display = init_script()

from src import plotting  # noqa: F401  (inline in notebooks, Agg when headless)


def load_data():
    """Load the modelling table and print a summary."""
    from src._bootstrap import load_modelling_table

    data = load_modelling_table()
    print(
        f"Modelling table: {data.shape[0]} rows x {data.shape[1]} cols "
        f"({data['year'].min()}-{data['year'].max()})"
    )
    return data

## 3.2 Data quality

No missing values and no duplicate years is the first thing to confirm.

In [2]:
def check_data_quality(data):
    """Inspect missing values, dtypes, and duplicates."""
    quality = c.pd.DataFrame(
        {
            "column": data.columns,
            "dtype": data.dtypes.astype(str).values,
            "n_nonnull": data.notna().sum().values,
            "n_missing": data.isna().sum().values,
            "unique": data.nunique().values,
        }
    )
    display(quality)
    print(f"Total missing values: {int(data.isna().sum().sum())}")
    print(f"Duplicate years: {int(data['year'].duplicated().sum())}")

## 3.3 Target distribution — UK wheat yield (t/ha)

In [3]:
def plot_target_distribution(data):
    """Histogram + KDE of wheat yields."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

    axes[0].hist(data["yield_t_ha"], bins=16, color="#1f77b4", edgecolor="white")
    axes[0].set_xlabel("Yield (t/ha)")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Histogram of UK wheat yield")

    data["yield_t_ha"].plot.kde(ax=axes[1], color="#d62728", linewidth=2)
    axes[1].axvline(
        data["yield_t_ha"].mean(),
        color="#2ca02c",
        linestyle="--",
        label=f"mean = {data['yield_t_ha'].mean():.2f}",
    )
    axes[1].set_xlabel("Yield (t/ha)")
    axes[1].set_ylabel("Density")
    axes[1].set_title("Kernel density estimate")
    axes[1].legend()

    fig.tight_layout()
    plt.show()


def yield_summary(data):
    """Descriptive statistics of the target variable."""
    return data["yield_t_ha"].describe().to_frame("UK wheat yield (t/ha)")

## 3.4 The yield time series (1980–2024)

A clear upward trend dominates, with notable dips. We annotate the events
that the modelling table encodes as dummy variables.

In [4]:
_ANNOTATIONS = {
    1992: "1992 CAP reform",
    2005: "2005 CAP reform",
    2020: "2020 wet autumn",
    2022: "2022 Ukraine shock",
}


def plot_yield_timeseries(data):
    """Yield over time with policy-event annotations."""
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(
        data["year"],
        data["yield_t_ha"],
        marker="o",
        markersize=3,
        linewidth=1.6,
        color="#1f77b4",
        label="UK wheat yield (t/ha)",
    )

    for year, label in _ANNOTATIONS.items():
        row = data[data["year"] == year]
        if len(row):
            ax.annotate(
                label,
                (year, row["yield_t_ha"].iloc[0]),
                xytext=(year - 6, row["yield_t_ha"].iloc[0] + 0.55),
                arrowprops=dict(arrowstyle="->", color="gray", lw=0.9),
                fontsize=8,
                color="#333333",
            )

    ax.set_xlabel("Harvest year")
    ax.set_ylabel("Yield (t/ha)")
    ax.set_title("UK wheat yield 1980–2024 (national average)")
    ax.legend()
    fig.tight_layout()
    plt.show()

**Non-stationarity evidence.** The series is not stationary — the mean drifts
up from ~6.4 t/ha in the 1980s to ~8 t/ha in the 2010s. (This is a gradual
trend, not an abrupt regime shift — the policy dummies below are one-year
pulses, not steps.) This motivates:

* differencing / stochastic trends in the ARIMA-style models;
* the **expanding-window** evaluation protocol (the training mean is not a
  valid forecast of a level-shifting target);
* trend-aware baselines in stage 05.

In [5]:
def decade_stats(data):
    """Yield statistics grouped by decade."""
    d = data.assign(decade=(data["year"] // 10) * 10)
    by_decade = d.groupby("decade")["yield_t_ha"].agg(["mean", "std", "min", "max"])
    return by_decade.round(3)

## 3.5 Weather covariates over time

Temperature and rainfall across the four phenological windows
(autumn, winter, spring, grain fill).

In [6]:
TEMP_COLS = ["autumn_temp", "winter_temp", "spring_temp", "grainfill_temp"]
RAIN_COLS = ["autumn_rain", "winter_rain", "spring_rain", "grainfill_rain"]
_PAL = ["#1f77b4", "#2ca02c", "#ff7f0e", "#d62728"]


def plot_weather_covariates(data):
    """Seasonal temperature and rainfall over time."""
    fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True)
    for col, color in zip(TEMP_COLS, _PAL):
        axes[0].plot(
            data["year"],
            data[col],
            label=col.replace("_", " ").title(),
            color=color,
            linewidth=1.4,
        )
    axes[0].set_ylabel("Temperature (°C)")
    axes[0].set_title("Seasonal temperature windows")
    axes[0].legend(ncol=4, fontsize=8)

    for col, color in zip(RAIN_COLS, _PAL):
        axes[1].plot(
            data["year"],
            data[col],
            label=col.replace("_", " ").title(),
            color=color,
            linewidth=1.4,
        )
    axes[1].set_ylabel("Rainfall (mm)")
    axes[1].set_xlabel("Harvest year")
    axes[1].set_title("Seasonal rainfall windows")
    axes[1].legend(ncol=4, fontsize=8)

    fig.tight_layout()
    plt.show()

## 3.6 Correlations with yield

Which covariates move with yield? A full correlation heatmap, followed by a
ranked bar of each covariate's correlation with the target.

In [7]:
def plot_correlations(data):
    """Correlation heatmap of the modelling table + ranked bar of covariates vs yield."""
    cov_cols = [c for c in data.columns if c not in ("year", "yield_t_ha", "decade")]
    fig, ax = plt.subplots(figsize=(8.5, 7))
    sns.heatmap(
        data[["yield_t_ha"] + cov_cols].corr(),
        annot=True,
        fmt=".2f",
        cmap="RdBu_r",
        center=0,
        square=True,
        ax=ax,
        annot_kws={"size": 7},
        cbar_kws={"shrink": 0.8},
    )
    ax.set_title("Correlation matrix (modelling table)")
    fig.tight_layout()
    plt.show()

    corr_with_yield = data[cov_cols].corrwith(data["yield_t_ha"]).sort_values()
    fig, ax = plt.subplots(figsize=(9, 3.6))
    colors = ["#2ca02c" if v > 0 else "#d62728" for v in corr_with_yield.values]
    corr_with_yield.plot.barh(ax=ax, color=colors)
    ax.set_xlabel("Correlation with yield (t/ha)")
    ax.set_title("Covariates ranked by correlation with UK wheat yield")
    ax.axvline(0, color="black", linewidth=0.8)
    fig.tight_layout()
    plt.show()

    return corr_with_yield.to_frame("corr_with_yield").round(3)

### Key EDA findings

1. **Strong upward trend** in yield — forecasting must account for a
   non-stationary target (differencing, trend-aware baselines).
2. **Linear correlations are weak overall** (all |r| < 0.38). The strongest
   signal is `spring_temp` (r = 0.378, warm springs help); the rainfall
   windows are consistently weak and slightly *negative* (r from −0.01 to
   −0.14), i.e. wetter years tend to be marginally poorer — but no single
   covariate moves yield on its own. Correlations are computed on the raw
   (trending) series, so part of each association reflects shared time
   trends rather than pure year-to-year agronomy.
3. **Policy dummies** (`cap_1992`, `cap_2005`, `ukraine_2022`) are one-year
   **pulse** indicators (1 in the event year, 0 elsewhere) for transient
   shocks — the CAP-reform years and the 2022 Ukraine invasion. They are not
   step changes: a pulse cannot encode a lasting regime shift, so the
   persistent drift is left to differencing and trend-aware baselines.
4. **No missing data, no duplicates** — the table is clean and ready for
   feature engineering and modelling.

These observations directly shape the modelling choices in stages 04–05.

## Run

In [8]:
def main():
    """Run the full EDA pipeline."""
    data = load_data()
    print()
    check_data_quality(data)
    print()
    plot_target_distribution(data)
    display(yield_summary(data))
    print()
    plot_yield_timeseries(data)
    display(decade_stats(data))
    print()
    plot_weather_covariates(data)
    display(plot_correlations(data))

In [ ]:
if __name__ == "__main__":
    main()

# 04 · Feature Engineering

**Goal.** Define and demonstrate every piece of feature construction the
forecasting models rely on, so stage 05 is purely about fitting and
comparing models.

**Input.** `data/processed/uk_wheat_modelling_table_1980_2024.csv`.

The corrected pipeline uses a deliberately small, interpretable feature set:

| Feature | Used by | Construction |
|---|---|---|
| 8 weather covariates (raw) | ARIMAX, Prophet, RF, XGBoost, hybrid | raw values, **no transformation** |
| 3 policy dummies | all covariate-driven models | one-hot events |
| **Forecast weather** for year `Y+h` | ARIMAX, Prophet, RF, XGBoost, hybrid | each covariate forecast `h` steps ahead with **ARIMA(1,0,0)** on its own history |
| ARIMAX covariate subset | ARIMAX | stepwise: drop highest p-value > 0.10, cap 5 covariates |
| Residual target | hybrid | `y - ARIMA fitted values`, modelled by XGBoost |

**Why forecast the weather?** To predict yield in year `Y+h` the model must
know the weather of year `Y+h`, which is itself unknown at forecast time.
The pipeline therefore *projects* every covariate forward with a simple
ARIMA(1,0,0) — an honest, reproducible proxy for "what the weather is likely
to be".

**Scaling is not used** in the corrected results (`standardise=False` in the
cross-validation): tree ensembles are scale-invariant and ARIMA-family
models are fit on raw units.

## 4.1 Setup

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import matplotlib.pyplot as plt
import warnings

from src._bootstrap import init_script, common_imports

c = common_imports()
display = init_script()

from src import plotting  # noqa: F401  (inline in notebooks, Agg when headless)
from src.features import forecast_exogenous
from src.models import arimax_stepwise_selection

warnings.filterwarnings("ignore")

## 4.2 The model-ready feature matrix

Stage 05 constructs, for every training window `[1980, origin]`, the feature
matrix `X` = the covariate columns, and the target `y` = `yield_t_ha`.

In [2]:
def load_data():
    """Load the modelling table and show a feature summary."""
    from src._bootstrap import load_modelling_table

    data = load_modelling_table()
    covariate_cols = c.config.COVARIATE_COLS
    print(
        f"Loaded modelling table: {data.shape[0]} rows; {len(covariate_cols)} covariates"
    )

    X_demo = data[covariate_cols]
    feature_summary = c.pd.DataFrame(
        {
            "feature": covariate_cols,
            "dtype": X_demo.dtypes.astype(str).values,
            "mean": X_demo.mean().round(3).values,
            "std": X_demo.std().round(3).values,
            "min": X_demo.min().round(3).values,
            "max": X_demo.max().round(3).values,
            "n_missing": X_demo.isna().sum().values,
        }
    )
    display(feature_summary)
    return data, covariate_cols

## 4.3 Exogenous forecasting — the core feature operation

For a horizon `h`, every covariate is forecast `h` years ahead using
`ARIMA(1,0,0)` fitted on the training window. Below we demonstrate this on
`autumn_rain` (the covariate most correlated with yield) for `h = 1..4`,
training on the 1980–2020 window.

In [3]:
def demo_exogenous_forecast(data):
    """Demonstrate exogenous forecasting for autumn_rain over a 4-year horizon."""
    train_demo = data[data["year"] <= 2020].reset_index(drop=True)
    h = 4
    fcst = forecast_exogenous(train_demo, ["autumn_rain"], h)["autumn_rain"]

    fig, ax = plt.subplots(figsize=(10, 3.8))
    ax.plot(
        train_demo["year"],
        train_demo["autumn_rain"],
        marker="o",
        markersize=3,
        linewidth=1.4,
        color="#1f77b4",
        label="autumn_rain (observed)",
    )
    future_years = c.np.arange(2021, 2021 + h)
    ax.plot(
        future_years,
        fcst,
        marker="s",
        linewidth=1.6,
        color="#d62728",
        label="ARIMA(1,0,0) forecast",
    )
    ax.set_xlabel("Harvest year")
    ax.set_ylabel("Autumn rainfall (mm)")
    ax.set_title("Exogenous covariate forecasting with ARIMA(1,0,0)")
    ax.legend()
    fig.tight_layout()
    plt.show()

    display(
        c.pd.DataFrame({"year": future_years, "autumn_rain_forecast": fcst.round(2)})
    )

## 4.4 Forecast all covariates for the next 4 years

This is exactly the table that an operational forecast for 2025–2028 would
need. It is produced purely from data in this notebook — no external inputs.

In [4]:
def forecast_all(data, covariate_cols):
    """Forecast every covariate 4 years ahead and display the table."""
    all_fcst = forecast_exogenous(data, covariate_cols, 4)
    forecast_table = c.pd.DataFrame(all_fcst)
    forecast_table.index = [2025, 2026, 2027, 2028]
    forecast_table.index.name = "year"
    display(forecast_table.round(2))
    return forecast_table

## 4.5 ARIMAX stepwise covariate selection

The ARIMAX model prunes covariates by refitting an ARIMA(1,0,0) with all
candidate covariates and iteratively removing the least significant one
(largest p-value) while any p-value exceeds 0.10, keeping at most five
covariates. Here is the selection for the full 1980–2024 sample.

In [5]:
def demo_stepwise(data, covariate_cols):
    """Demonstrate ARIMAX stepwise selection on the full sample."""
    selected, steps = arimax_stepwise_selection(
        data,
        covariate_cols,
        alpha=c.config.ARIMAX_ALPHA,
        max_cov=c.config.ARIMAX_MAX_COVARIATES,
    )
    print(f"ARIMAX selected covariates on the full sample: {selected}")
    print("p-values at each step (last step = final selection):")
    display(steps[-1].round(4).to_frame("p-value"))

**Note.** In the expanding-window protocol the selection runs *inside every
training window*, so the chosen subset changes year to year. The full-sample
result above is illustrative only — stage 05 reruns selection per origin.

## 4.6 What stage 05 receives

* **Feature matrix**: the 11 covariate columns (8 weather + 3 policy dummies).
* **Exogenous forecasts**: ARIMA(1,0,0) projections for horizons 1–4
  (used by ARIMAX, Prophet, RF, XGBoost and the hybrid).
* **ARIMAX subset**: per-window stepwise selection (cap 5, p > 0.10).
* **Hybrid target**: ARIMA residuals refit to the covariate matrix with
  XGBoost.

No data is hard-coded anywhere — every table and plot above is computed from
`data/processed/uk_wheat_modelling_table_1980_2024.csv`.

In [6]:
def main():
    """Run the full stage 04: feature summaries, forecasts, and ARIMAX demo."""
    data, covariate_cols = load_data()
    demo_exogenous_forecast(data)
    forecast_all(data, covariate_cols)
    demo_stepwise(data, covariate_cols)

In [ ]:
if __name__ == "__main__":
    main()

# 05 · Model — forecast comparison & statistical inference

**Goal.** Reproduce the Results chapter of the thesis (Tables 4.1–4.7,
Figure 4.3) under the **exact corrected protocol**:

* expanding-window CV from 2000 (seed 42), horizons 1–4;
* one model fit per training origin, shared across all horizons;
* ARIMA selected by AICc over p,q ∈ 0–3 (d=0);
* SARIMA seasonal order (p,0,q,1); ARIMAX stepwise (p > 0.10, cap 5);
* Prophet changepoint scale from {0.01, 0.05, 0.1} by 1-year-ahead CV;
* ML models tuned once on the full series with TimeSeriesSplit(5);
* exogenous covariates forecast with ARIMA(1,0,0).

**Input.** `data/processed/uk_wheat_modelling_table_1980_2024.csv`.

**Outputs** (saved to `data/outputs/`):

| Output | Thesis table |
|---|---|
| `model_comparison_results_corrected.csv` | Tables 4.1, 4.2, 4.6 |
| `baseline_results.csv` | Table 4.3 |
| `dm_test_results.csv` | Table 4.4 |
| `pi_coverage_results_corrected.csv` | Table 4.5 |
| `pi_detailed_results_corrected.csv` | Figure 4.3 |
| `oracle_exogenous_results.csv` | Table 4.7 |

The final section verifies every output against the canonical thesis numbers
stored in `data/expected/`.

Everything model-specific lives in `src/models.py`; this stage only
orchestrates the comparison, inference and verification.

## 5.1 Setup

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import logging
import time
import warnings

import matplotlib.pyplot as plt

from src._bootstrap import init_script, common_imports

c = common_imports()
display = init_script()

from src import plotting  # noqa: F401  (inline in notebooks, Agg when headless)
from src.cv import ExpandingWindowCV, evaluate_baseline
from src.metrics import rmse, mae, diebold_mariano
from src.guards import assert_balanced_test_sets, warn_on_swallowed_fits
from src.models import (
    persistence_factory,
    arima_factory,
    sarima_factory,
    arimax_factory,
    prophet_factory,
    tune_rf,
    tune_xgb,
    make_rf_factory,
    make_xgb_factory,
    make_hybrid_factory,
)

warnings.filterwarnings("ignore")
# Silence verbose statsmodels / Prophet / CmdStanPy logging up front. Prophet
# and CmdStanPy re-configure their loggers on import (e.g. forecaster.py sets
# its level to INFO), so the loggers are disabled here before they exist; a
# subsequent import only returns these same (disabled) logger objects.
for _name in (
    "cmdstanpy",
    "prophet",
    "prophet.models",
    "prophet.logger",
    "prophet.plot",
    "prophet.forecaster",
    "fbprophet",
    "numexpr",
    "numexpr.utils",
):
    logging.getLogger(_name).disabled = True
logging.basicConfig(level=logging.INFO, format="%(message)s")


MODEL_ORDER = [
    "Persistence",
    "ARIMA",
    "SARIMA",
    "ARIMAX",
    "Prophet",
    "RandomForest",
    "XGBoost",
    "ARIMA+XGBoost",
]

_BAR_COLOURS = [
    "#7f7f7f",
    "#1f77b4",
    "#2ca02c",
    "#9467bd",
    "#ff7f0e",
    "#8c564b",
    "#e377c2",
    "#d62728",
]

## 5.2 The evaluation protocol

```text
  origin 2000 .. 2023                     (expanding window)
     |-> fit model ONCE on [1980 .. origin]   (shared across horizons)
     |-> predict origin + 1, +2, +3, +4 years ahead
  metrics: RMSE, MAE per horizon
  DM test: MSE loss, HLN small-sample correction, two-sided t p-value
  seed 42 everywhere
```

`ExpandingWindowCV` (in `src/cv.py`) implements this identically to the
corrected pipeline, including the per-origin model cache and per-prediction
timing / memory tracking.

In [2]:
def run_cv(data, name, factory):
    """Run the expanding-window CV for one model and print its h=1 RMSE."""
    cv = ExpandingWindowCV(
        data=data,
        model_factory=factory,
        model_name=name,
        horizons=c.config.HORIZONS,
        seed=c.config.SEED,
    )
    t0 = time.time()
    summary = cv.evaluate()
    elapsed = time.time() - t0
    detail = c.pd.DataFrame(cv.results)
    h1 = summary[summary["horizon"] == 1]["rmse"].values[0]
    print(f"  {name:16s} RMSE h=1 = {h1:.4f}  ({elapsed:6.1f}s)")
    if cv.skipped_folds:
        per_h = {}
        for s in cv.skipped_folds:
            per_h[s["horizon"]] = per_h.get(s["horizon"], 0) + 1
        print(
            f"  {name:16s} WARNING: {len(cv.skipped_folds)} skipped fold(s) "
            f"by horizon {per_h} (n_test not comparable)"
        )
    return summary, detail

## 5.3 Baselines (Table 4.3)

Two trivial benchmarks under the same protocol: **Climatology** (training
window mean) and **Naive_RandomWalk** (last observed yield). No model should
be taken seriously if it cannot beat these.

In [3]:
def run_baselines(data):
    """Compute baseline RMSE/MAE per horizon and save to CSV."""
    rows = []
    for horizon in c.config.HORIZONS:
        for bname, fn in [
            ("Climatology", lambda tr: tr["yield_t_ha"].mean()),
            ("Naive_RandomWalk", lambda tr: tr["yield_t_ha"].iloc[-1]),
        ]:
            y_true, y_pred = evaluate_baseline(
                data, horizon, fn, initial_train_end=c.config.INITIAL_TRAIN_END
            )
            rows.append(
                [
                    bname,
                    horizon,
                    float(c.np.sqrt(c.np.mean((y_true - y_pred) ** 2))),
                    float(c.np.mean(c.np.abs(y_true - y_pred))),
                    len(y_true),
                ]
            )
    baseline_df = c.pd.DataFrame(
        rows, columns=["model", "horizon", "rmse", "mae", "n_test"]
    )
    baseline_df.to_csv(c.config.OUTPUT_FILES["baselines"], index=False)
    print(baseline_df.round(4).to_string(index=False))
    return baseline_df

## 5.4 Statistical models (Tables 4.1 / 4.2 / 4.6, part 1)

Persistence, ARIMA, SARIMA, ARIMAX and Prophet. This cell is the slowest of
the statistical block because Prophet runs an internal 1-year-ahead
changepoint-selection CV for every training window.

In [4]:
def run_statistical_models(data, all_summaries, all_details):
    """Run the five statistical model families under expanding-window CV."""
    for name, factory in [
        ("Persistence", persistence_factory),
        ("ARIMA", arima_factory),
        ("SARIMA", sarima_factory),
        ("ARIMAX", arimax_factory),
        ("Prophet", prophet_factory),
    ]:
        s, d = run_cv(data, name, factory)
        all_summaries.append(s)
        all_details.append(d)
    return all_summaries, all_details

## 5.5 Machine-learning models (Tables 4.1 / 4.2 / 4.6, part 2)

Hyperparameters are tuned **once** on the full series using
TimeSeriesSplit(5). The hybrid fits an ARIMA (AICc, d ∈ {0,1}) and then
models the residuals with XGBoost.

In [5]:
def run_ml_models(data, all_summaries, all_details):
    """Tune and evaluate RandomForest, XGBoost, and the ARIMA+XGBoost hybrid."""
    from sklearn.model_selection import TimeSeriesSplit
    import xgboost as xgb  # noqa: F401  (imported for side-effect in factory)

    X_full = data[c.config.COVARIATE_COLS].values
    y_full = data["yield_t_ha"].values
    tscv = TimeSeriesSplit(n_splits=c.config.TSCV_N_SPLITS)

    print("Tuning RandomForest...")
    rf_params = tune_rf(X_full, y_full, tscv)
    print(f"  RF params: {rf_params}")
    print("Tuning XGBoost...")
    xgb_params = tune_xgb(X_full, y_full, tscv)
    print(f"  XGB params: {xgb_params}")

    for name, factory in [
        ("RandomForest", make_rf_factory(rf_params)),
        ("XGBoost", make_xgb_factory(xgb_params)),
        ("ARIMA+XGBoost", make_hybrid_factory(xgb_params)),
    ]:
        s, d = run_cv(data, name, factory)
        all_summaries.append(s)
        all_details.append(d)

    return all_summaries, all_details, rf_params, xgb_params

## 5.6 Results — RMSE comparison (Tables 4.1, 4.2, 4.6)

All eight models, all four horizons, plus the per-fold detail file.

In [6]:
def aggregate_and_plot(comparison, details):
    """Save comparison tables, print the RMSE pivot, and plot grouped bars."""
    comparison.to_csv(c.config.OUTPUT_FILES["comparison"], index=False)
    details.to_csv(c.config.OUTPUT_FILES["details"], index=False)
    print(
        f"Saved {c.config.OUTPUT_FILES['comparison'].name} ({len(comparison)} rows) and "
        f"{c.config.OUTPUT_FILES['details'].name} ({len(details)} rows)"
    )

    pivot = comparison.pivot_table(index="model", columns="horizon", values="rmse")
    pivot = pivot.reindex(MODEL_ORDER)
    print("\n=== RMSE by model and horizon (t/ha) ===")
    print(pivot.round(4).to_string())

    fig, ax = plt.subplots(figsize=(10.5, 4.6))
    width = 0.09
    x = c.np.arange(len(MODEL_ORDER))
    for i, h in enumerate(c.config.HORIZONS):
        vals = pivot[h].values
        ax.bar(
            x + (i - 1.5) * width, vals, width, label=f"h = {h}", color=_BAR_COLOURS[i]
        )
    ax.set_xticks(x, MODEL_ORDER, rotation=20)
    ax.set_ylabel("RMSE (t/ha)")
    ax.set_title("Forecast accuracy by model and horizon (expanding-window CV)")
    ax.legend(ncol=4)
    fig.tight_layout()
    plt.show()

    print("RMSE by model and horizon, ranked (lower is better):")
    display(pivot.rank(ascending=True).astype(int).round(0))
    return pivot

## 5.7 Diebold–Mariano tests (Table 4.4)

Pairwise DM tests with MSE loss and the HLN small-sample correction across
the seven forecast models (Persistence excluded). A **negative** statistic
favours the model listed first.

In [7]:
def run_dm_tests(details):
    """Compute pairwise DM tests and save results."""
    dm_no_baseline = details[details["model"] != "Persistence"]
    dm_results = []
    for h in c.config.HORIZONS:
        models = sorted(dm_no_baseline["model"].unique())
        for i, m1 in enumerate(models):
            for m2 in models[i + 1 :]:
                h1 = dm_no_baseline[
                    (dm_no_baseline["model"] == m1) & (dm_no_baseline["horizon"] == h)
                ].sort_values("test_year")
                h2 = dm_no_baseline[
                    (dm_no_baseline["model"] == m2) & (dm_no_baseline["horizon"] == h)
                ].sort_values("test_year")
                common = c.pd.merge(
                    h1[["test_year", "y_true", "y_pred"]],
                    h2[["test_year", "y_true", "y_pred"]],
                    on="test_year",
                    suffixes=("_1", "_2"),
                )
                if len(common) < 2:
                    continue
                dm_stat, p_val = diebold_mariano(
                    common["y_true_1"].values,
                    common["y_pred_1"].values,
                    common["y_pred_2"].values,
                    loss="MSE",
                    h=h,
                )
                e1 = common["y_true_1"].values - common["y_pred_1"].values
                e2 = common["y_true_1"].values - common["y_pred_2"].values
                dm_results.append(
                    {
                        "horizon": h,
                        "model_1": m1,
                        "model_2": m2,
                        "loss": "MSE",
                        "dm_statistic": round(dm_stat, 4),
                        "p_value": round(p_val, 4),
                        "significant_005": bool(p_val < 0.05),
                        "significant_001": bool(p_val < 0.01),
                        "n_common": len(common),
                        "rmse_1": round(float(c.np.sqrt(c.np.mean(e1**2))), 4),
                        "rmse_2": round(float(c.np.sqrt(c.np.mean(e2**2))), 4),
                        "mae_1": round(float(c.np.mean(c.np.abs(e1))), 4),
                        "mae_2": round(float(c.np.mean(c.np.abs(e2))), 4),
                    }
                )
    dm_df = c.pd.DataFrame(dm_results)
    dm_df.to_csv(c.config.OUTPUT_FILES["dm_tests"], index=False)
    print(f"Saved {c.config.OUTPUT_FILES['dm_tests'].name} ({len(dm_df)} rows)")
    for h in c.config.HORIZONS:
        sub = dm_df[dm_df["horizon"] == h]
        print(
            f"  h={h}: {len(sub)} pairs, significant_005={sub['significant_005'].sum()}, "
            f"significant_001={sub['significant_001'].sum()}"
        )
    return dm_df

### DM heatmap (Figure: significance at h = 1)

The heatmap shows the DM statistic for every ordered pair at h = 1. Blue
cells mean the row model outperforms the column model (negative statistic).

In [8]:
def plot_dm_heatmap(dm_df):
    """Heatmap of DM statistics at h = 1."""
    dm_h1 = dm_df[dm_df["horizon"] == 1]
    models_h1 = sorted(dm_h1["model_1"].unique())
    stat_mat = c.pd.DataFrame(0.0, index=models_h1, columns=models_h1)
    for _, r in dm_h1.iterrows():
        stat_mat.loc[r["model_1"], r["model_2"]] = r["dm_statistic"]
        stat_mat.loc[r["model_2"], r["model_1"]] = -r["dm_statistic"]

    fig, ax = plt.subplots(figsize=(7, 5.6))
    im = ax.imshow(stat_mat.values, cmap="RdBu_r", vmin=-5, vmax=5)
    ax.set_xticks(range(len(models_h1)), models_h1, rotation=40, ha="right")
    ax.set_yticks(range(len(models_h1)), models_h1)
    ax.set_title("DM statistics at h = 1 (negative favours row model)")
    for i in range(len(models_h1)):
        for j in range(len(models_h1)):
            if i != j:
                ax.text(
                    j,
                    i,
                    f"{stat_mat.values[i, j]:.1f}",
                    ha="center",
                    va="center",
                    fontsize=8,
                )
    fig.colorbar(im, shrink=0.8)
    fig.tight_layout()
    plt.show()

## 5.8 Prediction-interval coverage (Table 4.5, Figure 4.3)

95% prediction intervals for ARIMA and Prophet under the corrected
specifications (`interval_width = 0.95`, exogenous covariates forecast with
ARIMA(1,0,0)). Coverage reports the fraction of out-of-sample years whose
true yield fell inside the interval.

Both predictors come straight from `src/models.py` — the same AICc selection
and Prophet changepoint selection as the main comparison, plus a
`predict_interval(h)` method — so this section can never drift from the
models it measures.

In [9]:
def compute_arima_pi(data):
    """Compute ARIMA 95% prediction intervals across all origins and horizons."""
    rows = []
    skipped = 0
    for train_end in range(c.config.INITIAL_TRAIN_END, int(data["year"].max())):
        train_df = data[data["year"] <= train_end]
        predictor = arima_factory(train_df, None)
        for h in c.config.HORIZONS:
            test_year = train_end + h
            if test_year > data["year"].max():
                continue
            try:
                y_pred, lo, hi = predictor.predict_interval(h)
            except Exception:
                skipped += 1
                continue
            test_row = data[data["year"] == test_year]
            if len(test_row) == 0:
                continue
            y_true = test_row["yield_t_ha"].values[0]
            rows.append(
                {
                    "model": "ARIMA",
                    "horizon": h,
                    "test_year": int(test_year),
                    "y_true": float(y_true),
                    "y_pred": y_pred,
                    "pi_lower": lo,
                    "pi_upper": hi,
                    "pi_width": hi - lo,
                    "covered": bool(lo <= y_true <= hi),
                }
            )
    return c.pd.DataFrame(rows), skipped


def compute_prophet_pi(data):
    """Compute Prophet 95% prediction intervals across all origins and horizons."""
    rows = []
    skipped = 0
    for train_end in range(c.config.INITIAL_TRAIN_END, int(data["year"].max())):
        train_df = data[data["year"] <= train_end]
        c.np.random.seed(c.config.SEED)
        predictor = prophet_factory(train_df, None)
        for h in c.config.HORIZONS:
            test_year = train_end + h
            if test_year > data["year"].max():
                continue
            try:
                y_pred, lo, hi = predictor.predict_interval(h)
            except Exception:
                skipped += 1
                continue
            test_row = data[data["year"] == test_year]
            if len(test_row) == 0:
                continue
            y_true = test_row["yield_t_ha"].values[0]
            rows.append(
                {
                    "model": "Prophet",
                    "horizon": h,
                    "test_year": int(test_year),
                    "y_true": float(y_true),
                    "y_pred": y_pred,
                    "pi_lower": lo,
                    "pi_upper": hi,
                    "pi_width": hi - lo,
                    "covered": bool(lo <= y_true <= hi),
                }
            )
    return c.pd.DataFrame(rows), skipped


def compute_prediction_intervals(data):
    """Compute PI coverage tables + chart for ARIMA and Prophet."""
    print("Computing ARIMA prediction intervals...")
    arima_pi, arima_skipped = compute_arima_pi(data)
    pi_attempted = len(arima_pi) + arima_skipped
    warn_on_swallowed_fits(pi_attempted, len(arima_pi), "ARIMA-PI")
    print(f"  ARIMA rows: {len(arima_pi)}, skipped: {arima_skipped}")

    print("Computing Prophet prediction intervals...")
    prophet_pi, prophet_skipped = compute_prophet_pi(data)
    pi_attempted = len(prophet_pi) + prophet_skipped
    warn_on_swallowed_fits(pi_attempted, len(prophet_pi), "Prophet-PI")
    print(f"  Prophet rows: {len(prophet_pi)}, skipped: {prophet_skipped}")

    pi_details = c.pd.concat([arima_pi, prophet_pi], ignore_index=True)
    pi_summary = []
    for model in ["ARIMA", "Prophet"]:
        mdf = pi_details[pi_details["model"] == model]
        for h in c.config.HORIZONS:
            sub = mdf[mdf["horizon"] == h]
            pi_summary.append(
                {
                    "model": model,
                    "horizon": h,
                    "pi_coverage_95": f"{sub['covered'].mean():.1%}",
                    "avg_pi_width": f"{sub['pi_width'].mean():.3f}",
                    "n_test": len(sub),
                }
            )
    pi_summary_df = c.pd.DataFrame(pi_summary)
    assert_balanced_test_sets(pi_details)
    print("  PI test-year symmetry across models: OK")
    pi_details.to_csv(c.config.OUTPUT_FILES["pi_details"], index=False)
    pi_summary_df.to_csv(c.config.OUTPUT_FILES["pi_coverage"], index=False)
    print("\nCoverage + width summary:")
    print(pi_summary_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(8.5, 4))
    for i, model in enumerate(["ARIMA", "Prophet"]):
        sub = pi_summary_df[pi_summary_df["model"] == model]
        cov = [float(v.strip("%")) / 100 for v in sub["pi_coverage_95"]]
        ax.plot(c.config.HORIZONS, cov, marker="o", label=model)
    ax.axhline(0.95, color="green", linestyle="--", label="nominal 95%")
    ax.set_ylim(0.5, 1.0)
    ax.set_xticks(c.config.HORIZONS)
    ax.set_xlabel("Horizon (years)")
    ax.set_ylabel("Coverage rate")
    ax.set_title("95% prediction-interval coverage by horizon")
    ax.legend()
    fig.tight_layout()
    plt.show()

    return pi_details, pi_summary_df

## 5.9 Oracle exogenous experiment (Table 4.7)

What if the models had **perfect knowledge** of the future weather instead of
the ARIMA(1,0,0) projections? We re-run the covariate-driven models feeding
the true covariate values (`oracle`) and report the RMSE improvement over the
standard forecasts. ARIMA needs no exogenous inputs, so its oracle rows carry
zero improvement (shown for context).

Instead of re-implementing the models, the factories accept an ``oracle_fc``
callable that replaces `features.forecast_exogenous` — so the oracle run uses
the exact same model code as the main comparison.

In [10]:
def oracle_forecast_factory(data):
    """Build an exogenous forecaster that returns the TRUE future values."""

    def oracle_fc(train_df, cov_cols, horizon):
        fc = {}
        origin = int(train_df["year"].max())
        for col in cov_cols:
            vals = []
            for k in range(1, horizon + 1):
                yr = origin + k
                row = data[data["year"] == yr]
                vals.append(
                    row[col].values[0] if len(row) > 0 else float(train_df[col].mean())
                )
            fc[col] = c.np.array(vals)
        return fc

    return oracle_fc


def run_oracle(data, rf_params, xgb_params):
    """Re-run covariate-driven models with perfect weather foresight; save table."""
    from functools import partial

    oracle_fc = oracle_forecast_factory(data)
    oracle_summaries = []
    for name, factory in [
        ("ARIMAX", partial(arimax_factory, oracle_fc=oracle_fc)),
        ("Prophet", partial(prophet_factory, oracle_fc=oracle_fc)),
        ("RandomForest", partial(make_rf_factory(rf_params), oracle_fc=oracle_fc)),
        ("XGBoost", partial(make_xgb_factory(xgb_params), oracle_fc=oracle_fc)),
        (
            "ARIMA+XGBoost",
            partial(make_hybrid_factory(xgb_params), oracle_fc=oracle_fc),
        ),
    ]:
        summary, _ = run_cv(data, name, factory)
        oracle_summaries.append(summary)

    oracle_combined = c.pd.concat(oracle_summaries, ignore_index=True).rename(
        columns={"rmse": "oracle_rmse"}
    )[["model", "horizon", "oracle_rmse", "n_test"]]

    standard = c.pd.read_csv(c.config.OUTPUT_FILES["comparison"])
    standard = standard[standard["model"] != "Persistence"]
    merged = standard.merge(
        oracle_combined[["model", "horizon", "oracle_rmse"]],
        on=["model", "horizon"],
        how="left",
    )
    merged["improvement_pct"] = (
        (merged["rmse"] - merged["oracle_rmse"]) / merged["rmse"] * 100
    ).round(1)
    merged = merged.rename(columns={"rmse": "standard_rmse"})

    # ARIMA needs no exog → it gets no oracle run, so its oracle_rmse is NaN after
    # the merge. Add context rows with oracle_rmse = standard_rmse (zero improvement)
    # for every horizon so the oracle table is complete for all 8 models × 4 horizons.
    for h in [1, 2, 3, 4]:
        row = standard[(standard["model"] == "ARIMA") & (standard["horizon"] == h)]
        if len(row) > 0:
            r = row.iloc[0]
            merged = c.pd.concat(
                [
                    merged,
                    c.pd.DataFrame(
                        [
                            {
                                "model": "ARIMA",
                                "horizon": h,
                                "standard_rmse": r["rmse"],
                                "oracle_rmse": r["rmse"],
                                "improvement_pct": 0.0,
                                "n_test": r["n_test"],
                            }
                        ]
                    ),
                ],
                ignore_index=True,
            )

    merged = merged.sort_values(["model", "horizon"]).reset_index(drop=True)
    cols = [
        "horizon",
        "model",
        "standard_rmse",
        "oracle_rmse",
        "improvement_pct",
        "n_test",
    ]
    merged = merged.dropna(subset=["oracle_rmse"]).reset_index(drop=True)
    merged[cols].to_csv(c.config.OUTPUT_FILES["oracle"], index=False)
    print(f"\nSaved {c.config.OUTPUT_FILES['oracle'].name} ({len(merged)} rows)")
    print(merged[cols].round(4).to_string(index=False))
    return merged

## 5.10 Verification against the canonical thesis numbers

Every output is compared with the values stored in `data/expected/` (extracted
from the thesis Results chapter). Small tolerances allow for library-version
drift; any "DIFF" means a thesis number was not reproduced.

In [11]:
def verify(fname, keys, vals, atol):
    """Compare an output CSV against the canonical expected file under tolerance."""
    e = c.pd.read_csv(c.config.EXPECTED_DIR / fname)
    p = c.pd.read_csv(c.config.OUTPUT_DIR / fname)
    key_e = set(map(tuple, e[keys].itertuples(index=False)))
    key_p = set(map(tuple, p[keys].itertuples(index=False)))
    if key_e != key_p:
        print(f"  {fname}: KEY MISMATCH (exp {len(key_e)} rows, got {len(key_p)})")
        return False
    m = e.merge(p, on=keys, suffixes=("_exp", "_got"))
    bad = []
    for v in vals:
        ec, gc = v + "_exp", v + "_got"
        if ec not in m or gc not in m:
            bad.append((v, "col missing"))
            continue
        col_exp, col_got = m[ec], m[gc]
        mask = ~(col_exp.isna() | col_got.isna())
        if c.pd.api.types.is_numeric_dtype(col_exp):
            diff = (col_exp[mask] - col_got[mask]).abs()
            n_bad = int((diff > atol).sum())
        else:
            n_bad = int((col_exp[mask] != col_got[mask]).sum())
        if n_bad:
            bad.append((v, n_bad))
    status = "OK" if not bad else f"DIFF {bad}"
    print(f"  {fname}: {status}")
    return not bad


def verify_outputs(details):
    """Run the full verification gate; return True if all checks pass."""
    all_ok = True
    print("Verifying outputs against the canonical numbers in data/expected/...")

    # F-4 guard: every model must be evaluated on the same test years per horizon.
    assert_balanced_test_sets(details)
    print("  test-year symmetry across models: OK")

    all_ok &= verify(
        "model_comparison_results_corrected.csv",
        ["model", "horizon"],
        ["rmse", "mae", "n_test"],
        1e-4,
    )
    all_ok &= verify(
        "dm_test_results.csv",
        ["horizon", "model_1", "model_2"],
        ["dm_statistic", "p_value", "n_common", "rmse_1", "rmse_2", "mae_1", "mae_2"],
        1e-4,
    )
    all_ok &= verify(
        "baseline_results.csv", ["model", "horizon"], ["rmse", "mae", "n_test"], 1e-6
    )
    all_ok &= verify(
        "oracle_exogenous_results.csv",
        ["horizon", "model"],
        ["standard_rmse", "oracle_rmse", "improvement_pct", "n_test"],
        1e-4,
    )
    all_ok &= verify(
        "pi_detailed_results_corrected.csv",
        ["model", "horizon", "test_year"],
        ["y_true", "y_pred", "pi_lower", "pi_upper", "pi_width"],
        1e-3,
    )

    mc = c.pd.read_csv(c.config.EXPECTED_DIR / "pi_coverage_results_corrected.csv").merge(
        c.pd.read_csv(c.config.OUTPUT_DIR / "pi_coverage_results_corrected.csv"),
        on=["model", "horizon"],
        suffixes=("_exp", "_got"),
    )
    cov_ok = (
        (mc["pi_coverage_95_exp"] == mc["pi_coverage_95_got"]).all()
        and (mc["avg_pi_width_exp"] == mc["avg_pi_width_got"]).all()
        and (mc["n_test_exp"] == mc["n_test_got"]).all()
    )
    print("  pi_coverage_results_corrected.csv:", "OK" if cov_ok else "DIFF")
    all_ok = all_ok and bool(cov_ok)

    print()
    if all_ok:
        print("ALL CHECKS PASSED - every canonical number was reproduced.")
    else:
        print("SOME CHECKS FAILED - inspect the DIFF details above.")
    return all_ok

## 5.11 Practitioner decision guide (deliverable, FR-9)

A compact, machine-generated summary for the practitioner. Every number below
is read directly from the verified output CSVs — nothing is hand-written, so
prose and data can never drift apart (NFR-10).

In [12]:
def generate_decision_guide():
    """Write a markdown decision guide from the verified output CSVs."""
    comparison_g = c.pd.read_csv(c.config.OUTPUT_FILES["comparison"])
    dm_g = c.pd.read_csv(c.config.OUTPUT_FILES["dm_tests"])
    pi_g = c.pd.read_csv(c.config.OUTPUT_FILES["pi_coverage"])
    oracle_g = c.pd.read_csv(c.config.OUTPUT_FILES["oracle"])

    lines = ["# UK wheat yield forecasting — practitioner decision guide"]
    lines.append("")
    lines.append(
        "Generated from the verified outputs of `scripts/05_Model.py` "
        "(see `data/expected/` for the verification gate)."
    )
    lines.append("")

    overall = comparison_g.groupby("model")["rmse"].mean().sort_values()
    lines.append(f"## Best overall model: {overall.index[0]}")
    lines.append("")
    lines.append("Mean RMSE across horizons 1-4 (t/ha):")
    lines.append("")
    lines.append("| Model | Mean RMSE |")
    lines.append("|---|---|")
    for model, v in overall.items():
        lines.append(f"| {model} | {v:.4f} |")
    lines.append("")

    lines.append("### Best model per horizon")
    lines.append("")
    lines.append("| Horizon | Model | RMSE (t/ha) |")
    lines.append("|---|---|---|")
    for h in c.config.HORIZONS:
        row = comparison_g[comparison_g["horizon"] == h].sort_values("rmse").iloc[0]
        lines.append(f"| h={h} | {row['model']} | {row['rmse']:.4f} |")
    lines.append("")

    wins = dm_g[dm_g["significant_005"]]
    lines.append(
        f"## Statistically significant pairwise differences (DM, 5%): {len(wins)} pairs"
    )
    lines.append("")
    lines.append("| Horizon | Pair | DM stat | p |")
    lines.append("|---|---|---|---|")
    for _, r in wins.sort_values("p_value").head(10).iterrows():
        lines.append(
            f"| h={r['horizon']} | {r['model_1']} vs {r['model_2']} "
            f"| {r['dm_statistic']:.2f} | {r['p_value']:.4f} |"
        )
    lines.append("")

    lines.append("## Prediction-interval coverage (nominal 95%)")
    lines.append("")
    lines.append("| Model | Horizon | Coverage | Avg width |")
    lines.append("|---|---|---|---|")
    for _, r in pi_g.iterrows():
        lines.append(
            f"| {r['model']} | h={r['horizon']} | {r['pi_coverage_95']} | {r['avg_pi_width']} |"
        )
    lines.append("")

    lines.append("## Ceiling: perfect weather foresight (oracle experiment)")
    lines.append("")
    best_oracle = oracle_g.loc[oracle_g["improvement_pct"].idxmax()]
    lines.append(
        f"Largest benefit: {best_oracle['model']} at h={best_oracle['horizon']} "
        f"({best_oracle['improvement_pct']:.1f}% RMSE reduction with perfect "
        "weather instead of ARIMA(1,0,0) projections)."
    )
    lines.append("")
    lines.append("| Model | Horizon | Std RMSE | Oracle RMSE | Impr. % |")
    lines.append("|---|---|---|---|---|")
    for _, r in (
        oracle_g.sort_values("improvement_pct", ascending=False).head(6).iterrows()
    ):
        lines.append(
            f"| {r['model']} | h={r['horizon']} | {r['standard_rmse']:.4f} "
            f"| {r['oracle_rmse']:.4f} | {r['improvement_pct']:.1f} |"
        )

    guide_path = c.config.OUTPUT_FILES["decision_guide"]
    guide_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(
        f"Decision guide written to {guide_path.name} ({guide_path.stat().st_size:,} bytes)"
    )

## 5.12 Summary of saved outputs

In [13]:
def print_output_summary():
    """Print the size of every output file that was written."""
    print("Outputs written to data/outputs/:")
    for key, path in c.config.OUTPUT_FILES.items():
        if path.exists():
            print(f"  {path.name:50s} {path.stat().st_size:>10,} bytes")

    print("\nAll charts are rendered inline in this notebook (no PNG files are saved).")

## main()

In [14]:
def main():
    """Run the full stage 05: model comparison, inference, PIs, oracle, verify."""
    from src._bootstrap import load_modelling_table

    data = load_modelling_table()
    print(
        f"Loaded modelling table: {data.shape[0]} rows "
        f"({data['year'].min()}-{data['year'].max()})"
    )

    run_baselines(data)

    all_summaries, all_details = [], []
    run_statistical_models(data, all_summaries, all_details)
    all_summaries, all_details, rf_params, xgb_params = run_ml_models(
        data, all_summaries, all_details
    )

    comparison = c.pd.concat(all_summaries, ignore_index=True)
    details = c.pd.concat(all_details, ignore_index=True)
    aggregate_and_plot(comparison, details)

    dm_df = run_dm_tests(details)
    plot_dm_heatmap(dm_df)

    compute_prediction_intervals(data)

    run_oracle(data, rf_params, xgb_params)

    all_ok = verify_outputs(details)

    generate_decision_guide()

    print_output_summary()

    if not all_ok:
        c.sys.exit(1)

In [ ]:
if __name__ == "__main__":
    main()